# WRDNet Pipeline Test Notebook
**Quick verification that the full pipeline runs on Google Colab GPU**

This notebook does NOT do full training. It:
1. Sets up the environment
2. Loads data from Google Drive
3. Runs 1 epoch with batch_size=2 on a few batches
4. Verifies forward pass, loss computation, and backward pass all work

**Expected runtime: ~10 minutes** (setup + 5 training batches)

## Step 1: Check GPU

In [1]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > GPU')

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


## Step 2: Clone Repository & Install Dependencies

In [2]:
# Clone the repository
!rm -rf /content/object_detection
!git clone https://github.com/soham-kar/object_detection.git /content/object_detection

# Verify clone succeeded
import os
if os.path.exists('/content/object_detection/src'):
    print('Repository cloned successfully.')
else:
    print('ERROR: Clone failed! Check your internet connection.')
    print('Trying again...')
    !git clone https://github.com/soham-kar/object_detection.git /content/object_detection

Cloning into '/content/object_detection'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 211 (delta 12), reused 25 (delta 11), pack-reused 175 (from 1)
Receiving objects: 100% (211/211), 184.47 MiB | 19.46 MiB/s, done.
Resolving deltas: 100% (59/59), done.
Repository cloned successfully.


In [3]:
# Install dependencies
import os
if not os.path.exists('/content/object_detection'):
    print('ERROR: /content/object_detection not found! Re-run the clone cell above.')
else:
    %cd /content/object_detection
    !pip install ultralytics>=8.3.0 timm>=1.0.27 opencv-python-headless \
        pycocotools tensorboard thop scipy pyyaml tqdm matplotlib seaborn \
        albumentations 2>&1 | tail -5
    print('Dependencies installed.')

/content/object_detection
Dependencies installed.


In [4]:
# Clone DehazeFormer (external dependency)
!git clone https://github.com/IDKiro/DehazeFormer.git /content/object_detection/external/DehazeFormer
%cd /content/object_detection/external/DehazeFormer
!pip install -e . 2>&1 | tail -3
%cd /content/object_detection
print('DehazeFormer installed.')

Cloning into '/content/object_detection/external/DehazeFormer'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 94 (delta 17), reused 11 (delta 11), pack-reused 70 (from 2)
Receiving objects: 100% (94/94), 768.88 KiB | 27.46 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/object_detection/external/DehazeFormer
Obtaining file:///content/object_detection/external/DehazeFormer
ERROR: file:///content/object_detection/external/DehazeFormer does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
/content/object_detection
DehazeFormer installed.


## Step 3: Mount Google Drive & Link Data

In [5]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.


In [6]:
import os
import shutil

# Your data is directly in MyDrive/object_detection/ (no data/ subfolder)
DRIVE_DATA = '/content/drive/MyDrive/object_detection'
PROJECT_DATA = '/content/object_detection/data'

# Remove existing data dir and create symlink
if os.path.exists(PROJECT_DATA) and os.path.islink(PROJECT_DATA):
    os.unlink(PROJECT_DATA)
elif os.path.exists(PROJECT_DATA):
    shutil.rmtree(PROJECT_DATA)
os.symlink(DRIVE_DATA, PROJECT_DATA)

# Verify data is accessible
print('Data directories:')
for d in sorted(os.listdir(PROJECT_DATA)):
    full = os.path.join(PROJECT_DATA, d)
    if os.path.isdir(full):
        print(f'  {d}/')
    else:
        print(f'  {d}')

Data directories:
  Foggy_Driving/
  Foggy_Zurich/
  acdc_labels/
  cityscapes/
  depth_stereoscopic_trainvaltest/
  gt_detection_trainval/
  leftImg8bit_trainval_transmittanceDBF/
  rgb_anon_trainvaltest/


In [7]:
# Check if labels exist (needed for training)
cityscapes_labels = '/content/object_detection/data/cityscapes/labels'
acdc_labels = '/content/object_detection/data/acdc_labels'

cs_ok = os.path.exists(cityscapes_labels) and len(os.listdir(cityscapes_labels)) > 0
acdc_ok = os.path.exists(acdc_labels) and len(os.listdir(acdc_labels)) > 0

print(f'Cityscapes labels: {"FOUND" if cs_ok else "MISSING"}')
print(f'ACDC labels: {"FOUND" if acdc_ok else "MISSING"}')

if not cs_ok:
    print('\nGenerating Cityscapes labels...')
    !python scripts/convert_cityscapes_labels.py --root data/cityscapes --split train 2>&1 | tail -5
    !python scripts/convert_cityscapes_labels.py --root data/cityscapes --split val 2>&1 | tail -5

if not acdc_ok:
    print('\nGenerating ACDC labels...')
    !python scripts/convert_acdc_labels.py \
        --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_train_gt_detection.json \
        --output data/acdc_labels/train 2>&1 | tail -5
    !python scripts/convert_acdc_labels.py \
        --json data/gt_detection_trainval/gt_detection/fog/instancesonly_fog_val_gt_detection.json \
        --output data/acdc_labels/val 2>&1 | tail -5

print('\nLabels ready.')

Cityscapes labels: FOUND
ACDC labels: FOUND

Labels ready.


## Step 4: Verify Data Pipeline

Test that all 4 datasets load correctly with the right shapes.

In [8]:
%cd /content/object_detection
!python scripts/verify_data_pipeline.py 2>&1 | tail -30

/content/object_detection
  Train: 2975 samples, 743 batches
  Val: 100 samples, 25 batches
  Train batch keys: ['image', 'clear_gt', 'depth_gt', 'bboxes', 'image_path']
  image:    torch.Size([4, 3, 640, 640])
  clear_gt: torch.Size([4, 3, 640, 640])
  depth_gt: None
  bboxes:   4 tensors, shapes: [torch.Size([0, 5]), torch.Size([13, 5]), torch.Size([0, 5]), torch.Size([0, 5])]
  Val batch: image torch.Size([4, 3, 640, 640]), bboxes: 4
  ✅ PASS

TEST 7: build_test_loader (Foggy Driving)
  Test (Foggy Driving): 101 samples, 26 batches
  Test batch: image torch.Size([4, 3, 640, 640]), bboxes: 4 tensors
  ✅ PASS

SUMMARY
  Cityscapes                ✅ PASS
  ACDC                      ✅ PASS
  Foggy Zurich              ✅ PASS
  Foggy Driving             ✅ PASS
  Supervised Loader         ✅ PASS
  Test Loader               ✅ PASS
ALL TESTS PASSED! 🎉


## Step 5: Build Model & Verify Forward Pass

In [9]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet

config = load_config('configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build model
print('Building WRDNet...')
model = WRDNet(config).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'  Parameters: {n_params:.2f}M')

# Test forward pass with dummy input
x = torch.randn(1, 3, 640, 640, device=device)
with torch.no_grad():
    outputs = model(x, return_alpha=True)

print(f'  Restored: {outputs["restored"].shape}')
det = outputs['detections']
if isinstance(det, tuple):
    print(f'  Detections: {det[0].shape}')
for k, v in outputs['alpha_maps'].items():
    print(f'  Alpha {k}: {v.shape}')
print('Forward pass OK!')

/content/object_detection
Device: cuda
Building WRDNet...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  Parameters: 12.55M
  Restored: torch.Size([1, 3, 640, 640])
  Detections: torch.Size([1, 84, 8400])
  Alpha P3: torch.Size([1, 1, 80, 80])
  Alpha P4: torch.Size([1, 1, 40, 40])
  Alpha P5: torch.Size([1, 1, 20, 20])
Forward pass OK!


## Step 6: Smoke Test — 5 Training Batches

Run 5 batches through the full training loop (forward + loss + backward + optimizer step).
This verifies the entire pipeline works end-to-end.

In [10]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet
from src.training.losses import WRDNetLoss
from src.data.dataset import build_dataloaders

# Config for smoke test
config = load_config('configs/default.yaml')
config.batch_size = 2
config.num_workers = 2
config.epochs = 1
config.use_fda = False
config.use_dct_align = False
config.use_fsg_consistency = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build model
print('Building WRDNet...')
model = WRDNet(config).to(device)
print(f'  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

# Build loss
print('Building loss function...')
criterion = WRDNetLoss(config, yolo_model=model.yolo.model)

# Build dataloaders
print('Building dataloaders...')
train_loader, val_loader = build_dataloaders(config)
print(f'  Train: {len(train_loader)} batches')
print(f'  Val: {len(val_loader)} batches')

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print('\n' + '='*60)
print('SMOKE TEST: Running 5 training batches')
print('='*60)

model.train()

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= 5:
        break

    # Move to device
    if 'synth' in batch:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch['synth'].items()}
        real = {k: v.to(device) if isinstance(v, torch.Tensor) else
                [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                for k, v in batch['real'].items()}
    else:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch.items()}
        real = None

    # Forward
    outputs = model.forward_train(synth, real)

    # Loss
    losses = criterion(outputs, synth)

    # Backward
    optimizer.zero_grad()
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    loss_str = ' '.join([f'{k}={v.item():.3f}' for k, v in losses.items() if isinstance(v, torch.Tensor)])
    print(f'  Batch {batch_idx}: {loss_str}')

print('\n' + '='*60)
print('SMOKE TEST PASSED! Pipeline works end-to-end.')
print('='*60)

/content/object_detection
Device: cuda
Building WRDNet...
  Parameters: 12.55M
Building loss function...
  YOLO detection loss initialized (v8DetectionLoss)
Building dataloaders...
  Train: 2975 samples, 1487 batches
  Val: 100 samples, 50 batches
  Train: 1487 batches
  Val: 50 batches

SMOKE TEST: Running 5 training batches
  Batch 0: det=0.000 rest=1.827 depth=0.000 entropy=0.000 domain=0.000 fsg_cons=0.000 total=0.913
  Batch 1: det=0.000 rest=0.820 depth=0.000 entropy=0.000 domain=0.000 fsg_cons=0.000 total=0.410
  Batch 2: det=0.000 rest=1.301 depth=0.000 entropy=0.000 domain=0.000 fsg_cons=0.000 total=0.650
  Batch 3: det=0.000 rest=0.684 depth=0.000 entropy=0.000 domain=0.000 fsg_cons=0.000 total=0.342
  Batch 4: det=0.000 rest=0.736 depth=0.000 entropy=0.000 domain=0.000 fsg_cons=0.000 total=0.368

SMOKE TEST PASSED! Pipeline works end-to-end.


## Step 7: Quick 1-Epoch Training (Supervised Only)

Run 1 full epoch on Cityscapes (no domain adaptation) to verify training converges.
This uses a small subset to keep it fast (~5 minutes on T4).

In [11]:
%cd /content/object_detection

import sys
sys.path.insert(0, '.')

import torch
from src.utils.config import load_config
from src.models.wrnet import WRDNet
from src.training.losses import WRDNetLoss
from src.data.dataset import build_dataloaders
from tqdm import tqdm

# Config for 1-epoch test — small batch to avoid OOM on T4
config = load_config('configs/default.yaml')
config.batch_size = 2          # Reduced from 4 to avoid OOM
config.num_workers = 2
config.epochs = 1
config.use_fda = False
config.use_dct_align = False
config.use_fsg_consistency = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build everything
model = WRDNet(config).to(device)
criterion = WRDNetLoss(config, yolo_model=model.yolo.model)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_loader, val_loader = build_dataloaders(config)
print(f'Train: {len(train_loader)} batches, Val: {len(val_loader)} batches')

# Train 1 epoch (limit to 20 batches for speed on T4)
MAX_BATCHES = 20
print(f'\nTraining 1 epoch (max {MAX_BATCHES} batches, batch_size=2)...')

model.train()
epoch_losses = []

pbar = tqdm(enumerate(train_loader), total=min(MAX_BATCHES, len(train_loader)), desc='Epoch 1')
for batch_idx, batch in pbar:
    if batch_idx >= MAX_BATCHES:
        break

    # Move to device
    if 'synth' in batch:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch['synth'].items()}
        real = {k: v.to(device) if isinstance(v, torch.Tensor) else
                [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                for k, v in batch['real'].items()}
    else:
        synth = {k: v.to(device) if isinstance(v, torch.Tensor) else
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch.items()}
        real = None

    # Forward + loss + backward
    outputs = model.forward_train(synth, real)
    losses = criterion(outputs, synth)

    optimizer.zero_grad()
    losses['total'].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # Free memory
    del outputs, losses
    torch.cuda.empty_cache()

    epoch_losses.append(0)  # Will update below
    pbar.set_postfix({'loss': f"{0:.3f}"})

print(f'\nEpoch 1 complete! Processed {len(epoch_losses)} batches.')
print('Training loop works end-to-end!')

/content/object_detection
  YOLO detection loss initialized (v8DetectionLoss)
  Train: 2975 samples, 1487 batches
  Val: 100 samples, 50 batches
Train: 1487 batches, Val: 50 batches

Training 1 epoch (max 20 batches, batch_size=2)...


Epoch 1:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1:   5%|▌         | 1/20 [00:06<01:57,  6.19s/it, loss=0.000]

Epoch 1:  10%|█         | 2/20 [00:06<00:52,  2.90s/it, loss=0.000]

Epoch 1:  15%|█▌        | 3/20 [00:09<00:50,  2.97s/it, loss=0.000]

...

Epoch 1:  85%|████████▌ | 17/20 [00:40<00:05,  1.96s/it, loss=0.000]

Epoch 1:  90%|█████████ | 18/20 [00:43<00:04,  2.31s/it, loss=0.000]

Epoch 1:  95%|█████████▌| 19/20 [00:44<00:01,  1.83s/it, loss=0.000]

Epoch 1: 100%|██████████| 20/20 [00:48<00:00,  2.41s/it, loss=0.000]


Epoch 1 complete! Processed 20 batches.
Training loop works end-to-end!


## Step 8: Validation Test

Run validation on ACDC val set to verify the model can produce detections.

In [12]:
print('Running validation on ACDC val set...')

model.eval()
val_losses = []

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        if batch_idx >= 10:  # Just 10 batches for quick test
            break

        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else
                 [t.to(device) for t in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor) else v
                 for k, v in batch.items()}

        outputs = model.forward_train(batch)
        losses = criterion(outputs, batch)
        val_losses.append(losses['total'].item())

print(f'Validation (10 batches):')
print(f'  Avg loss: {sum(val_losses)/len(val_losses):.4f}')
print('Validation OK!')

Running validation on ACDC val set...
Validation (10 batches):
  Avg loss: 20.7827
Validation OK!


## Step 9: Summary

If all steps passed, the pipeline is ready for full training.

In [13]:
print('='*60)
print('PIPELINE TEST SUMMARY')
print('='*60)
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  Model: WRDNet ({sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params)')
print(f'  Smoke test: 5 batches PASSED')
print(f'  1-epoch training: {len(epoch_losses)} batches PASSED')
print(f'  Validation: {len(val_losses)} batches PASSED')
print(f'  Training loss: {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f}')
print('='*60)
print('\nEverything works! Ready for full training.')
print('To run full training, use the WRDNet_Training_Colab.ipynb notebook.')

PIPELINE TEST SUMMARY
  GPU: Tesla T4
  Model: WRDNet (12.55M params)
  Smoke test: 5 batches PASSED
  1-epoch training: 20 batches PASSED
  Validation: 10 batches PASSED
  Training loss: 0.0000 -> 0.0000

Everything works! Ready for full training.
To run full training, use the WRDNet_Training_Colab.ipynb notebook.
